<font color='darkorange'> Unless otherwise noted, **this notebook will not be reviewed or autograded.**</font> You are welcome to use it for scratchwork, but **only the files listed in the exercises will be checked.**

---

# Exercises

For these exercises, add your functions to the *apputil\.py* file and *app\.py* file as instructed. *These exercises use the same [Titanic dataset](https://www.kaggle.com/competitions/titanic/data) as the lab.*

## Loading the Dataset

When you load your data for this exercise, **make sure to rename columns into lowercase-underscore format.** Write a routine that would rename text like "Age Group" into "age_group". For example, in this data, the column "Pclass" should be renamed to "pclass". *This is a good habit to get into going forward!*


## Exercise 1: Survival Patterns


In [2]:
import plotly.express as px
import pandas as pd
import re

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/leontoddjohnson/datasets/main/data/titanic.csv')

def to_snake_case(name):
    name = name.replace(' ', '_')
    name = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '_', name)
    return name.lower()

df.columns = [to_snake_case(col) for col in df.columns]

df.head()

,passenger_id,survived,pclass,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


For this exercise you will analyze survival patterns on the Titanic by looking at passenger class, sex, and age group. Name the function `survival_demographics()` (with no arguments).

1. Create a new column in the Titanic dataset that classifies passengers into age categories (i.e., a pandas `category` series). The categories (case sensitive) should be:
    - `"Child"` (up to 12)
    - `"Teen"` (13–19)
    - `"Adult"` (20–59)
    - `"Senior"` (60+)  
  
	Hint: The `pd.cut()` function might come in handy here.

2. Group the passengers by class, sex, and age group.  

3. For each group, calculate:  
    - The total number of passengers, `n_passengers`
    - The number of survivors, `n_survivors`
    - The survival rate, `survival_rate`

4. Return a DataFrame that includes the results for *all* combinations of class, sex, and age group.  

5. Order the results so they are easy to interpret.  

6. Come up with a clear question that your results table makes you curious about (e.g., “Did women in first class have a higher survival rate than men in other classes?”). Write this question in your `app.py` file above the call to your visualization function, using `st.write("Your Question Here")`.
   
7. Create a Plotly visualization in a function named `visualize_demographic()` that directly addresses your question by returning a Plotly figure (e.g., `fig = px. ...`). You are free to choose the chart type that you think best communicates the findings. Be creative — try different approaches, compare them, and ensure that your chart clearly answers the question you posed.


In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def survival_demographics():
    age_df = df.dropna(subset=['age']).copy()

    bins = [0, 12, 19, 59, np.inf]
    labels = ['Child', 'Teen', 'Adult', 'Senior']
    age_df['age_group'] = pd.cut(age_df['age'], bins=bins, labels=labels, include_lowest=True)
    age_df['age_group'] = pd.Categorical(age_df['age_group'], categories=labels, ordered=True)

    result = (
        age_df.groupby(['pclass', 'sex', 'age_group'], observed=False)
        .agg(n_passengers=('survived', 'size'), n_survivors=('survived', 'sum'))
        .reset_index()
    )
    result['survival_rate'] = result['n_survivors'] / result['n_passengers']
    result = result.sort_values(['pclass', 'sex', 'age_group']).reset_index(drop=True)

    return result


def visualize_demographic():
    # Question: How does survival rate vary across passenger class, sex, and age
    # group, and which of these three factors seems to matter most?
    data = survival_demographics().copy()
    data['class_label'] = data['pclass'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})
    data['sex_label'] = data['sex'].str.title()

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Approach 1: Survival Rate by Class & Sex',
            'Approach 2: Survival Rate Heatmap (Sex + Age Group vs Class)',
            'Approach 3: Survival Rate Trend by Age Group',
            'Approach 4: Group Size (bubble) vs Survival Rate (color)',
        ),
    )

    # Approach 1: grouped bar chart, survival rate by class & sex
    # (aggregated across age groups)
    agg_class_sex = (
        data.groupby(['class_label', 'sex_label'])
        .apply(lambda g: g['n_survivors'].sum() / g['n_passengers'].sum(), include_groups=False)
        .reset_index(name='survival_rate')
    )
    colors = {'Male': '#4C78A8', 'Female': '#F58518'}
    for sex_label, subset in agg_class_sex.groupby('sex_label'):
        fig.add_trace(
            go.Bar(
                x=subset['class_label'], y=subset['survival_rate'],
                name=sex_label, marker_color=colors[sex_label],
                legendgroup=sex_label,
            ),
            row=1, col=1,
        )

    # Approach 2: heatmap of survival rate, (sex + age group) x class
    data['group_label'] = data['sex_label'] + ' - ' + data['age_group'].astype(str)
    heatmap_data = data.pivot_table(index='group_label', columns='class_label', values='survival_rate')
    fig.add_trace(
        go.Heatmap(
            z=heatmap_data.values, x=heatmap_data.columns, y=heatmap_data.index,
            colorscale='RdYlGn', zmin=0, zmax=1, showscale=True,
            colorbar=dict(title='Survival Rate', len=0.4, y=0.8),
        ),
        row=1, col=2,
    )

    # Approach 3: line chart, survival rate trend across age groups,
    # one line per class/sex combination
    for (class_label, sex_label), grp in data.groupby(['class_label', 'sex_label']):
        grp = grp.sort_values('age_group')
        fig.add_trace(
            go.Scatter(
                x=grp['age_group'].astype(str), y=grp['survival_rate'],
                mode='lines+markers', name=f'{class_label} - {sex_label}',
                legendgroup=f'{class_label}-{sex_label}', showlegend=False,
            ),
            row=2, col=1,
        )

    # Approach 4: bubble chart, bubble size = group size, color = survival rate
    fig.add_trace(
        go.Scatter(
            x=data['class_label'], y=data['group_label'],
            mode='markers',
            marker=dict(
                size=data['n_passengers'], sizemode='area',
                sizeref=2. * data['n_passengers'].max() / (40. ** 2), sizemin=4,
                color=data['survival_rate'], colorscale='RdYlGn', cmin=0, cmax=1,
                showscale=False,
            ),
            showlegend=False,
            text=data['n_passengers'],
            hovertemplate='n_passengers=%{text}<br>survival_rate=%{marker.color:.2f}<extra></extra>',
        ),
        row=2, col=2,
    )

    fig.update_yaxes(range=[0, 1], title_text='Survival Rate', row=1, col=1)
    fig.update_yaxes(range=[0, 1], title_text='Survival Rate', row=2, col=1)
    fig.update_xaxes(title_text='Age Group', row=2, col=1)
    fig.update_layout(
        height=800, width=1000,
        title_text='Titanic Survival Demographics: Four Views of Class, Sex, and Age Group',
        barmode='group',
    )

    return fig


survival_table = survival_demographics()
survival_table
    

,pclass,sex,age_group,n_passengers,n_survivors,survival_rate
0,1,female,Child,1,0,0.000000
1,1,female,Teen,13,13,1.000000
2,1,female,Adult,68,66,0.970588
3,1,female,Senior,3,3,1.000000
4,1,male,Child,3,3,1.000000
5,1,male,Teen,4,1,0.250000
6,1,male,Adult,80,34,0.425000
7,1,male,Senior,14,2,0.142857
8,2,female,Child,8,8,1.000000
9,2,female,Teen,8,8,1.000000


## Exercise 2: Family Size and Wealth

Using the Titanic dataset, write a function named `family_groups()` to explore the relationship between family size, passenger class, and ticket fare.  

1. Create a new column in the Titanic dataset that represents the total family size for each passenger, `family_size`. Family size is defined as the number of siblings/spouses aboard plus the number of parents/children aboard, plus the passenger themselves.

2. Group the passengers by family size and passenger class. For each group, calculate:  
   - The total number of passengers, `n_passengers`
   - The average ticket fare, `avg_fare`
   - The minimum and maximum ticket fares (to capture variation in wealth), `min_fare` and `max_fare`

3. Return a table with these results, sorted so that the values are clear and easy to interpret (for example, by class and then family size).

4. Write a function called `last_names()` that extracts the last name of each passenger from the `Name` column, and returns the count for each last name (i.e., a pandas series with last name as index, and count as value). Does this result agree with that of the data table above? Share your findings in your app using `st.write`.

5. Just like you did in Exercise 1, come up with a clear question that your results makes you curious about. Write this question in your app.py file above the call to your visualization function. Then, create a Plotly visualization in a function named `visualize_families()` that directly addresses your question. As in Exercise 1 you are free to choose the chart type that you think best communicates the findings.

## Bonus Question

Add a new column, `older_passenger`, to the Titanic dataset that indicates whether each passenger’s age is above the median age for *their* passenger class. So, suppose row $x$ is in passenger class 2. Then, a value of `True` at row $x$ would indicate that passenger older than 50% of class 2 passengers, and `False` would indicate that they younger.

- You should use pandas functions to accomplish this.
- The new column should contain Boolean values (True if the age is above the median, False if less than or equal to).
- Return the updated table in the function `determine_age_division()`

Once you’ve created this column, consider how this age division relates to your analysis above. Try to visualize this analysis in Plotly using the function name `visualize_age_division()`.